In [1]:
import pandas as pd

# Load dataset
df = pd.read_csv('credit_risk_dataset.csv')

# Show first 5 rows
df.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000.0,16.02,1.0,0.59,Y,3.0
1,21,9600,OWN,5.0,EDUCATION,B,1000.0,11.14,0.0,0.10,N,2.0
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500.0,12.87,1.0,0.57,N,3.0
3,23,65500,RENT,4.0,MEDICAL,C,35000.0,15.23,1.0,0.53,N,2.0
4,24,54400,RENT,8.0,MEDICAL,C,35000.0,14.27,1.0,0.55,Y,4.0


In [2]:
# Check dataset shape
print(df.shape)

# Column names
print(df.columns)

# Dataset information
df.info()

(19083, 12)
Index(['person_age', 'person_income', 'person_home_ownership',
       'person_emp_length', 'loan_intent', 'loan_grade', 'loan_amnt',
       'loan_int_rate', 'loan_status', 'loan_percent_income',
       'cb_person_default_on_file', 'cb_person_cred_hist_length'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19083 entries, 0 to 19082
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   person_age                  19083 non-null  int64  
 1   person_income               19083 non-null  int64  
 2   person_home_ownership       19083 non-null  object 
 3   person_emp_length           18522 non-null  float64
 4   loan_intent                 19083 non-null  object 
 5   loan_grade                  19082 non-null  object 
 6   loan_amnt                   19082 non-null  float64
 7   loan_int_rate               17288 non-null  float64
 8   loan_status         

In [4]:
# Check missing values
df.isnull().sum()

,0
person_age,0
person_income,0
person_home_ownership,0
person_emp_length,561
loan_intent,0
loan_grade,1
loan_amnt,1
loan_int_rate,1795
loan_status,1
loan_percent_income,1


In [5]:
# Fill numerical missing values with median

df['loan_int_rate'].fillna(df['loan_int_rate'].median(), inplace=True)

df['person_emp_length'].fillna(df['person_emp_length'].median(), inplace=True)

/tmp/ipykernel_8589/1690089058.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['loan_int_rate'].fillna(df['loan_int_rate'].median(), inplace=True)
/tmp/ipykernel_8589/1690089058.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value

In [6]:
df.isnull().sum()

,0
person_age,0
person_income,0
person_home_ownership,0
person_emp_length,0
loan_intent,0
loan_grade,1
loan_amnt,1
loan_int_rate,0
loan_status,1
loan_percent_income,1


In [7]:
# Convert categorical columns into numbers

df = pd.get_dummies(df, drop_first=True)

In [8]:
# Features
X = df.drop('loan_status', axis=1)

# Target
y = df['loan_status']

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [12]:
from sklearn.linear_model import LogisticRegression
import numpy as np
from sklearn.preprocessing import StandardScaler

# Create model
lr_model = LogisticRegression(max_iter=5000) # Increased max_iter

# Identify rows with NaN in X_train
nan_in_X_train = X_train.isnull().any(axis=1)
# Identify rows with NaN in y_train
nan_in_y_train = y_train.isnull()

# Get the boolean mask for rows to keep (no NaNs in X_train or y_train)
rows_to_keep = ~(nan_in_X_train | nan_in_y_train)

# Filter X_train and y_train to remove rows with NaNs
X_train_cleaned = X_train[rows_to_keep]
y_train_cleaned = y_train[rows_to_keep]

# Scale the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_cleaned)
X_test_scaled = scaler.transform(X_test) # Transform X_test using the scaler fitted on X_train

# Train model
lr_model.fit(X_train_scaled, y_train_cleaned)

LogisticRegression(max_iter=5000)

In [13]:
y_pred_lr = lr_model.predict(X_test)

# Probability predictions for ROC-AUC
y_prob_lr = lr_model.predict_proba(X_test)[:, 1]

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


In [14]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

print("Logistic Regression Results")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Precision:", precision_score(y_test, y_pred_lr))
print("Recall:", recall_score(y_test, y_pred_lr))
print("F1-Score:", f1_score(y_test, y_pred_lr))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_lr))

Logistic Regression Results
Accuracy: 0.632957820277705
Precision: 0.09443507588532883
Recall: 0.06086956521739131
F1-Score: 0.07402511566424323
ROC-AUC: 0.42422708649127283


In [15]:
print(classification_report(y_test, y_pred_lr))

              precision    recall  f1-score   support

         0.0       0.73      0.81      0.77      2897
         1.0       0.09      0.06      0.07       920

    accuracy                           0.63      3817
   macro avg       0.41      0.44      0.42      3817
weighted avg       0.58      0.63      0.60      3817



In [17]:
from sklearn.ensemble import RandomForestClassifier

# Create model
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Train model
rf_model.fit(X_train_cleaned, y_train_cleaned)

RandomForestClassifier(random_state=42)

In [18]:
y_pred_rf = rf_model.predict(X_test)

# Probability predictions for ROC-AUC
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

In [19]:
print("Random Forest Results")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall:", recall_score(y_test, y_pred_rf))
print("F1-Score:", f1_score(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_rf))

Random Forest Results
Accuracy: 0.9316216924286088
Precision: 0.9673758865248226
Recall: 0.741304347826087
F1-Score: 0.8393846153846154
ROC-AUC: 0.9404336945265718


In [20]:
print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

         0.0       0.92      0.99      0.96      2897
         1.0       0.97      0.74      0.84       920

    accuracy                           0.93      3817
   macro avg       0.95      0.87      0.90      3817
weighted avg       0.93      0.93      0.93      3817



In [21]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest'],
    'Precision': [
        precision_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_rf)
    ],
    'Recall': [
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_rf)
    ],
    'F1-Score': [
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_rf)
    ],
    'ROC-AUC': [
        roc_auc_score(y_test, y_prob_lr),
        roc_auc_score(y_test, y_prob_rf)
    ]
})

print(results)

                 Model  Precision    Recall  F1-Score   ROC-AUC
0  Logistic Regression   0.094435  0.060870  0.074025  0.424227
1        Random Forest   0.967376  0.741304  0.839385  0.940434


In [22]:
results.style.highlight_max(axis=0)

,Model,Precision,Recall,F1-Score,ROC-AUC
0,Logistic Regression,0.094435,0.060870,0.074025,0.424227
1,Random Forest,0.967376,0.741304,0.839385,0.940434
